# Policyscope quickstart: свои данные (RU)

Цель: быстро запустить OPE на собственном DataFrame через официальный high-level API `compare_policies(...)` и корректно прочитать результат.

## 1) Импорты и тихий режим

В примерах ниже мы подавляем шумный логинг, чтобы итоговый notebook было удобно читать на GitHub.

In [1]:
import pathlib
import sys

cwd = pathlib.Path.cwd()
for cand in [cwd / 'src', cwd.parent / 'src', pathlib.Path('/workspace/OffPolicyLab/src')]:
    if cand.exists() and str(cand) not in sys.path:
        sys.path.insert(0, str(cand))

import io
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import pandas as pd

from policyscope.data import BanditSchema, LoggedBanditDataset
from policyscope.comparison import compare_policies

logging.getLogger().setLevel(logging.WARNING)
pd.set_option('display.max_columns', 50)

def quiet_call(fn, *args, **kwargs):
    out, err = io.StringIO(), io.StringIO()
    with redirect_stdout(out), redirect_stderr(err):
        return fn(*args, **kwargs)

## 2) Минимальный copy-adapt-run шаблон для собственного DataFrame

Ниже — минимальный путь: подставьте свой `df`, схему и объект policy B с методом `action_probs(df) -> (n, k)`.

In [2]:
# Замените на свой источник данных
np.random.seed(7)
n = 1500
df = pd.DataFrame({
    'user_id': np.arange(n),
    'loyal': np.random.binomial(1, 0.45, size=n),
    'age': np.random.randint(18, 70, size=n),
    'risk': np.random.uniform(0.0, 1.0, size=n),
    'income': np.random.lognormal(mean=10.4, sigma=0.5, size=n),
    'a_A': np.random.choice([0, 1, 2, 3], size=n),
})

# Пример target (ваша колонка награды может называться иначе)
logit = -0.4 + 0.8 * df['loyal'] - 0.6 * df['risk'] + 0.1 * (df['a_A'] == 1)
p = 1.0 / (1.0 + np.exp(-logit))
df['accept'] = np.random.binomial(1, p)

# Если у вас есть logged propensity, храните её в отдельной колонке
df['propensity_A'] = np.random.uniform(0.05, 0.95, size=n)

# Шаг 1: формализуем контракт данных
schema = BanditSchema(
    action_col='a_A',
    reward_col='accept',
    feature_cols=['loyal', 'age', 'risk', 'income'],
    cluster_col='user_id',
    propensity_col='propensity_A',  # уберите, если в логах нет propensity
)
logged = LoggedBanditDataset(df=df, schema=schema)
logged.df.head(3)

,user_id,loyal,age,risk,income,a_A,accept,propensity_A
0,0,0,48,0.327615,51719.740551,1,1,0.794312
1,1,1,56,0.477093,23390.953646,3,1,0.728735
2,2,0,25,0.856267,38361.096794,0,1,0.624362


## 3) Пример policy B

В реальном кейсе policy B обычно приходит из вашей модели/ранжировщика.
Для демо определим простую стохастическую политику на основе простых score-функций.

In [3]:
class HeuristicPolicyB:
    def action_probs(self, x: pd.DataFrame) -> np.ndarray:
        s0 = 0.4 + 0.5 * x['loyal'] - 0.3 * x['risk']
        s1 = 0.2 + 0.2 * (x['age'] < 30) + 0.4 * (x['risk'] < 0.4)
        s2 = 0.3 + 0.3 * (x['income'] > x['income'].median())
        s3 = 0.2 + 0.2 * (x['age'] > 55)
        raw = np.vstack([s0, s1, s2, s3]).T.astype(float)
        raw = raw - raw.max(axis=1, keepdims=True)
        e = np.exp(raw)
        return e / e.sum(axis=1, keepdims=True)

policyB = HeuristicPolicyB()

## 4) Официальный high-level запуск `compare_policies(...)`

Практический default: начинайте с `estimator='dr'`, `propensity_source='auto'`.

In [4]:
summary = quiet_call(
    compare_policies,
    logged,
    policyB,
    estimator='dr',
    target='accept',
    n_boot=20,
    alpha=0.1,
    weight_clip=20.0,
    tau=20.0,
    propensity_source='auto',
)
res = summary.to_dict()
pd.Series({
    'V_A': res['V_A'],
    'V_B': res['V_B'],
    'Delta': res['Delta'],
    'Delta_CI': res.get('Delta_CI'),
    'p_value': res.get('p_value'),
    'trust_level': res.get('trust_level'),
    'diagnostic_warnings': ', '.join(res.get('diagnostic_warnings', [])) or '-',
    'propensity_source': res.get('propensity_source'),
})

V_A                                                            0.424
V_B                                                         0.433351
Delta                                                       0.009351
Delta_CI               (-0.008300593672744267, 0.025531678327243852)
p_value                                                     0.571429
trust_level                                                  caution
diagnostic_warnings                                                -
propensity_source                                             logged
dtype: object

## 5) Что делать, если logged propensity есть / нет

- `propensity_source='auto'` (рекомендуется): использует logged propensity при наличии валидной колонки, иначе fallback на оценку behavior-model.
- `propensity_source='logged'`: строгий режим только по логам (ошибка, если колонка отсутствует/невалидна).
- `propensity_source='estimated'`: всегда оценивать propensity моделью.

In [5]:
modes = []
for mode in ['auto', 'logged', 'estimated']:
    try:
        s = quiet_call(
            compare_policies,
            logged,
            policyB,
            estimator='ips',
            target='accept',
            n_boot=10,
            alpha=0.1,
            propensity_source=mode,
        )
        d = s.to_dict()
        modes.append({'mode': mode, 'V_B': d['V_B'], 'Delta': d['Delta'], 'propensity_source_used': d.get('propensity_source')})
    except Exception as e:
        modes.append({'mode': mode, 'error': str(e)})

pd.DataFrame(modes)

,mode,V_B,Delta,propensity_source_used
0,auto,0.368853,-0.055147,logged
1,logged,0.368853,-0.055147,logged
2,estimated,0.419748,-0.004252,estimated


## 6) Как читать результат ответственно

- `V_A`, `V_B`: оценка policy value для logging policy A и candidate policy B.
- `Delta = V_B - V_A`: ожидаемая разница.
- `Delta_CI`: диапазон правдоподобных значений для Delta при выбранном bootstrap-инференсе.
- `p_value`: сигнал против нулевой гипотезы `Delta=0`; **не** мера размера эффекта.
- `diagnostics` + `diagnostic_warnings`: проверка, где OPE может быть нестабильным (overlap/ESS/weight tails).
- `trust_level`: сводный индикатор риска интерпретации (`ok`, `caution`, `elevated_concern`), но не «гарантия безопасности».

Подробнее: `docs/how_to_interpret_ope_outputs_ru.md`.